In [1]:
import pandas as pd
import numpy as np
import datetime

from pathlib import Path 

In [ ]:
current_dir = Path(Path.cwd()).parent
data_dir = current_dir / "data"
source_data_dir = data_dir / "raw"
print(data_dir)

d:\pedro\Programacao\.Projetos\dengue_prediction\dengue_prediction\data


In [3]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column_name(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

## Dengue data

In [4]:
def clear_cs_dengue_df(df, year):
    df.columns = [normalize_column_name(col) for col in df.columns]
    
    # rename
    if year < 2021:
        df = df.rename(columns={
            "nu_notificacao": "id",
            "dt_notificacao": "data",
        })
    else:
        df = df.rename(columns={
            "nu_notific": "id",
            "dt_notific": "data",
        })
    df = df[["id", "data"]]

    # remove rows with missing data
    df = df.dropna(subset=["data"])

    # data type
    if year < 2015:
        df["data"] = pd.to_datetime(
            df["data"],
            format="%Y/%m/%d %H:%M:%S",
            errors="coerce"
        )

        # verify if time is always same hour
        if df["data"].dt.time.unique() == [datetime.time(0, 0)]:
            df["data"] = pd.to_datetime(
                df["data"],
                format="%Y/%m/%d",
                errors="coerce"
            )
    elif year < 2021 or year in [2022, 2023]:
        df["data"] = pd.to_datetime(
            df["data"],
            format="%Y-%m-%d",
            errors="coerce"
        )
    else: 
        df["data"] = pd.to_datetime(
            df["data"],
            format="%d/%m/%Y",
            errors="coerce"
        )
    
    # remove duplicate 
    df = df.drop_duplicates(["id"])

    return df 

In [5]:
# INMET data for recife only go until 2021
dfs_years = list(range(2013, 2022))

dengue_dir = source_data_dir / "dengue_data/Recife"

dengue_df = pd.DataFrame()

for year in dfs_years:
    print(f"Ano: {year}")
    path = dengue_dir / f"casos-de-dengue-{year}.csv"
    if year in [2015, 2019]:
        df = pd.read_csv(path, sep=",")
    else:
        df = pd.read_csv(path, sep=";")
    df = clear_cs_dengue_df(df, year)
    dengue_df = pd.concat([dengue_df, df], ignore_index=True)



Ano: 2013
Ano: 2014
Ano: 2015


C:\Users\pedro\AppData\Local\Temp\ipykernel_17664\3634264432.py:12: DtypeWarning: Columns (30,52,96,112) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=",")


Ano: 2016


C:\Users\pedro\AppData\Local\Temp\ipykernel_17664\3634264432.py:14: DtypeWarning: Columns (54,112) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";")


Ano: 2017
Ano: 2018
Ano: 2019
Ano: 2020
Ano: 2021


In [6]:

print("df.info()\n")
print(dengue_df.info())
print("\nvalores unicos do bairro:")
print("shape:", dengue_df.shape)
print(f"porcentagem de valores ausentes:\n{(dengue_df.isnull().mean() * 100).sort_values(ascending=False)}%")
dengue_df[-10:-1]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82607 entries, 0 to 82606
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   id      82607 non-null  object        
 1   data    82607 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(1)
memory usage: 1.3+ MB
None

valores unicos do bairro:
shape: (82607, 2)
porcentagem de valores ausentes:
id      0.0
data    0.0
dtype: float64%


,id,data
82597,4173313,2021-12-29
82598,4200206,2021-12-29
82599,4033700,2021-12-29
82600,4193410,2021-12-29
82601,4195267,2021-12-29
82602,3975954,2021-12-30
82603,4033703,2021-12-30
82604,4208998,2021-12-30
82605,4087324,2021-12-31


### INMET (weather data)

In [7]:

def clear_weather_df(df, year):
    df.columns = [normalize_column_name(col) for col in df.columns]

    # dont have this columns in all data sources 
    for col in ["vento_rajada_maxima_m_s", "vento_direcao_horaria_gr_gr", "vento_velocidade_horaria_m_s"]:
        if col in df.columns:
            df = df.drop(columns=col)

    # rename coluns 
    df = df.rename(columns={
        "data_yyyy_mm_dd": "data",
        "hora_utc": "hora"
    })
    # remove rows with missing data
    df = df.dropna(subset=["data"])

    # data and hour type 
    if year < 2019:
        df["hora"] = df["hora"].str.split(":").str[0].astype(int)
        df["data"] = pd.to_datetime(
            df["data"],
            format="%Y-%m-%d",
            errors="coerce"
        )
    else:
        df["hora"] = (
            df.iloc[:, 1]
            .astype(str)
            .str.extract(r"(\d{2})\d{2}\s*UTC")[0]
            .astype("Int64")
        )
        df["data"] = pd.to_datetime(
            df["data"],
            format="%Y/%m/%d",
            errors="coerce"
        )
        
    # -9999 is the nan value of INMET 
    df = df.replace(-9999, np.nan)
    # remove columns with more than 30% of missing data
    df = df.loc[
        :, df.isnull().mean() < 0.3
    ]
    # complete missing values with sliding window median
    # ps: i dont think it causes data leakage 
    cols = [col for col in df.columns if col not in ["data", "hora"]]
    df[cols] = df[cols].fillna(
        df[cols].rolling(window=6, min_periods=2).median()
    )
    # remove the lines with more than 40% of missing data
    df = df.loc[
        df.isnull().mean(axis=1) < 0.6
    ]
    return df


In [8]:
weather_dir = source_data_dir / "weather_data"
weather_df = pd.DataFrame()

for year in dfs_years:
    print(f"Ano: {year}")
    df = weather_dir / f"INMET_NE_PE_A301_RECIFE_01-01-{year}_A_31-12-{year}.csv"
    df = pd.read_csv(df, sep=";", encoding="latin1", skiprows=8, decimal=",")
    df = clear_weather_df(df, year)
    weather_df = pd.concat([weather_df, df], ignore_index=True)

Ano: 2013


Ano: 2014
Ano: 2015
Ano: 2016
Ano: 2017
Ano: 2018
Ano: 2019
Ano: 2020
Ano: 2021


In [9]:

print("df.info()\n")
print(weather_df.info())
print("shape:", weather_df.shape)
print(f"\nporcentagem de valores ausentes:\n{(weather_df.isnull().mean() * 100).sort_values(ascending=False)}")
weather_df.head()



df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76363 entries, 0 to 76362
Data columns (total 15 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   data                                                76363 non-null  datetime64[ns]
 1   hora                                                76363 non-null  Int64         
 2   precipitacao_total_horario_mm                       76308 non-null  float64       
 3   pressao_atmosferica_ao_nivel_da_estacao_horaria_mb  76363 non-null  float64       
 4   pressao_atmosferica_max_na_hora_ant_aut_mb          76351 non-null  float64       
 5   pressao_atmosferica_min_na_hora_ant_aut_mb          76351 non-null  float64       
 6   temperatura_do_ar_bulbo_seco_horaria_c              76363 non-null  float64       
 7   temperatura_do_ponto_de_orvalho_c                   75871 non-null  float64       


,data,hora,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria
0,2013-01-01,0,0.0,1012.4,1012.4,1012.1,26.5,22.1,26.7,26.5,22.2,21.2,77.0,72.0,77.0
1,2013-01-01,1,0.0,1012.4,1012.5,1012.3,26.4,20.9,26.7,26.4,22.1,20.7,77.0,70.0,72.0
2,2013-01-01,2,0.0,1012.1,1012.5,1012.1,26.3,21.9,26.5,26.3,21.9,21.0,76.0,72.0,76.0
3,2013-01-01,3,0.6,1011.5,1012.1,1011.5,25.1,22.5,26.4,24.8,22.6,21.9,86.0,76.0,86.0
4,2013-01-01,4,0.0,1010.9,1011.5,1010.9,25.4,22.0,25.6,25.0,22.5,21.9,86.0,81.0,82.0


## Dataset 
- input(info_30_dias) 
- output(casos_total_dia_31)

In [10]:
cases_per_day = dengue_df.groupby(dengue_df["data"]).size().reset_index(name="contagem")
cases_per_day


,data,contagem
0,2013-01-03,3
1,2013-01-04,3
2,2013-01-05,2
3,2013-01-07,8
4,2013-01-08,4
...,...,...
2843,2021-12-27,12
2844,2021-12-28,6
2845,2021-12-29,5
2846,2021-12-30,3


In [16]:
# Maybe some inf about the weather change would be good 

mean_30_days_arlyer = pd.DataFrame()
# mean of each day
days_mean = weather_df.groupby(weather_df["data"]).mean()

for col in days_mean.columns:
    # shift(1) to dont get the current day, and rolling(30) to get the mean of the last 30 days
    # using only the 2013 we dont have the previous 30 days, so we will get NaN for the first 30 days, but we can use the 2012 data to fill these values later
    mean_30_days_arlyer[col] = days_mean[col].shift(1).rolling(window=30).mean()

# merge 2 sources 
dataset = pd.merge(mean_30_days_arlyer, cases_per_day, on="data")
dataset = dataset.dropna() # remove the lines with missing data, included the first 30 days of 2013 
dataset = dataset.rename(columns={"contagem": "casos_dengue"})
print(f"\nporcentagem de valores ausentes:\n{(dataset.isnull().mean() * 100).sort_values(ascending=False)}%")


porcentagem de valores ausentes:
data                                                  0.0
hora                                                  0.0
precipitacao_total_horario_mm                         0.0
pressao_atmosferica_ao_nivel_da_estacao_horaria_mb    0.0
pressao_atmosferica_max_na_hora_ant_aut_mb            0.0
pressao_atmosferica_min_na_hora_ant_aut_mb            0.0
temperatura_do_ar_bulbo_seco_horaria_c                0.0
temperatura_do_ponto_de_orvalho_c                     0.0
temperatura_maxima_na_hora_ant_aut_c                  0.0
temperatura_minima_na_hora_ant_aut_c                  0.0
temperatura_orvalho_max_na_hora_ant_aut_c             0.0
temperatura_orvalho_min_na_hora_ant_aut_c             0.0
umidade_rel_max_na_hora_ant_aut                       0.0
umidade_rel_min_na_hora_ant_aut                       0.0
umidade_relativa_do_ar_horaria                        0.0
casos_dengue                                          0.0
dtype: float64%


In [17]:
dataset.to_csv(data_dir/"processed/datasets/recife_dataset.csv", index=False)